# WeightWatcher `get_ESD`: robust plots and manual top-eigenvalue power-law fits

This notebook deliberately **does not use WeightWatcher's ESD plotting code**.

It:

1. loads the MuonClip checkpoint;
2. runs `WeightWatcher.analyze(plot=False)` only to initialize the layer analysis;
3. calls `watcher.get_ESD(layer=layer_id)` explicitly for every available layer;
4. lets you choose a `LAYER_ID` and an exact number of largest eigenvalues to remove;
5. plots the full ESD with occupancy-colored logarithmic bins, a log-space KDE, and an unbinned CCDF;
6. fits the **entire retained ESD** with the Python `powerlaw` MLE package by fixing `xmin` to the smallest retained eigenvalue;
7. also runs an automatic-`xmin` fit to locate any smaller power-law tail;
8. compares `power_law` with `truncated_power_law`; and
9. sweeps through removing 0, 1, 2, ... top eigenvalues.

The one cell marked **EDIT THIS CELL** is the normal control point.

In [ ]:
from pathlib import Path
import json
import os
import sys
import warnings

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import pandas as pd
import powerlaw
from scipy.stats import gaussian_kde
import torch
from IPython.display import display
import weightwatcher as ww

# Locate baseline/nanogpt_one_head whether Jupyter was launched from the
# repository root, the experiment directory, or its notebooks directory.
cwd = Path.cwd().resolve()
root_candidates = [
    cwd,
    cwd.parent,
    cwd / "baseline" / "nanogpt_one_head",
]
EXPERIMENT_ROOT = next(
    (
        candidate
        for candidate in root_candidates
        if (candidate / "configs" / "reference.yaml").is_file()
    ),
    None,
)
if EXPERIMENT_ROOT is None:
    raise FileNotFoundError(
        "Run from the rg_optimizers repository root, "
        "baseline/nanogpt_one_head, or its notebooks directory."
    )

sys.path.insert(0, str(EXPERIMENT_ROOT / "src"))
from rg_nanogpt_one_head.model import GPT, GPTConfig
from rg_nanogpt_one_head.spectral import (
    WeightMatrixHolder,
    _attach_matrix_metadata,
)

# This is the experiment discussed in the analysis. An exported RUN_DIR
# overrides the default, so the notebook remains reusable.
DEFAULT_RUN_DIR = Path(
    "/tmp/rg-nanogpt-long-muonclip-50ep/results/muon_clip/seed_1337"
)
RUN_DIR = Path(
    os.environ.get("RUN_DIR", str(DEFAULT_RUN_DIR))
).expanduser().resolve()

# Reproduce the checkpoint shown in the existing epoch-5.75 figure.
# Set TARGET_STEP=None to use the latest checkpoint.
TARGET_STEP = 56152
TARGET_EPOCH = None

MIN_EVALS = 20

print("experiment root:", EXPERIMENT_ROOT)
print("run directory:", RUN_DIR)
print("WeightWatcher version:", getattr(ww, "__version__", "unknown"))
print("powerlaw version:", getattr(powerlaw, "__version__", "unknown"))

## Load the selected checkpoint

`TARGET_STEP` takes precedence. If it is `None`, `TARGET_EPOCH` is used; if both are `None`, the latest checkpoint is loaded.

In [ ]:
if not RUN_DIR.is_dir():
    raise FileNotFoundError(f"Run directory does not exist: {RUN_DIR}")

manifest_path = RUN_DIR / "manifest.json"
metrics_path = RUN_DIR / "epoch_metrics.csv"
if not manifest_path.is_file() or not metrics_path.is_file():
    raise FileNotFoundError(
        f"{RUN_DIR} must contain manifest.json and epoch_metrics.csv"
    )

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
epoch_metrics = pd.read_csv(metrics_path)

for column in ("step", "nominal_epoch", "epoch"):
    if column in epoch_metrics.columns:
        epoch_metrics[column] = pd.to_numeric(
            epoch_metrics[column],
            errors="coerce",
        )

epoch_metrics = (
    epoch_metrics
    .dropna(subset=["step", "nominal_epoch"])
    .sort_values("step")
)

if TARGET_STEP is not None:
    matching = epoch_metrics.loc[
        epoch_metrics["step"].astype(int) == int(TARGET_STEP)
    ]
    if matching.empty:
        available = epoch_metrics["step"].astype(int).tolist()
        raise ValueError(
            f"TARGET_STEP={TARGET_STEP} is unavailable. "
            f"Available steps include: {available[-20:]}"
        )
    selected_checkpoint_row = matching.iloc[-1]
elif TARGET_EPOCH is not None:
    nearest_index = (
        epoch_metrics["nominal_epoch"] - float(TARGET_EPOCH)
    ).abs().idxmin()
    selected_checkpoint_row = epoch_metrics.loc[nearest_index]
else:
    selected_checkpoint_row = epoch_metrics.iloc[-1]

STEP = int(selected_checkpoint_row["step"])
NOMINAL_EPOCH = float(selected_checkpoint_row["nominal_epoch"])
ACTUAL_EPOCH = float(selected_checkpoint_row.get("epoch", np.nan))

checkpoint_path = Path(
    str(selected_checkpoint_row.get("checkpoint_path", ""))
)
if not checkpoint_path.is_file():
    matches = sorted(
        (RUN_DIR / "epoch_checkpoints").glob(
            f"*step_{STEP:07d}.pt"
        )
    )
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Could not resolve checkpoint for step {STEP}; "
            f"found {matches}"
        )
    checkpoint_path = matches[0]

payload = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False,
)
model = GPT(GPTConfig(**manifest["model"]))
model.load_state_dict(payload["model"])
model.eval()

holder = WeightMatrixHolder(model)
OUTPUT_DIR = (
    RUN_DIR
    / "diagnostics"
    / f"manual_get_esd_powerlaw_step_{STEP:07d}"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("checkpoint:", checkpoint_path)
print(
    f"step={STEP} nominal_epoch={NOMINAL_EPOCH:.3f} "
    f"actual_epoch={ACTUAL_EPOCH:.6f}"
)
print("output directory:", OUTPUT_DIR)

## Run WeightWatcher once, then call `get_ESD` layer by layer

The loop below is the important data-extraction step. Each `layer_id` is passed directly to:

```python
watcher.get_ESD(layer=layer_id)
```

Every extracted spectrum is saved as its own CSV.

In [ ]:
watcher = ww.WeightWatcher(model=holder)

details_raw = watcher.analyze(
    ERG=False,
    randomize=False,
    plot=False,
    min_evals=MIN_EVALS,
)
if details_raw is None or len(details_raw) == 0:
    raise RuntimeError("WeightWatcher returned no analyzable layers")

details = _attach_matrix_metadata(
    pd.DataFrame(details_raw),
    holder.matrix_metadata,
)

layer_columns = [
    column
    for column in (
        "layer_id",
        "matrix_name",
        "matrix_type",
        "M",
        "N",
        "alpha",
        "sigma",
        "D",
        "xmin",
        "xmax",
    )
    if column in details.columns
]
layer_table = (
    details[layer_columns]
    .sort_values("layer_id")
    .reset_index(drop=True)
)
display(layer_table)

ESD_BY_LAYER = {}
LAYER_NAME_BY_ID = {}

for row in layer_table.itertuples(index=False):
    layer_id = int(row.layer_id)
    matrix_name = str(row.matrix_name)

    # Explicit WeightWatcher ESD extraction, one layer at a time.
    spectrum = np.asarray(
        watcher.get_ESD(layer=layer_id),
        dtype=float,
    ).reshape(-1)
    spectrum = np.sort(
        spectrum[
            np.isfinite(spectrum)
            & (spectrum > 0)
        ]
    )

    if spectrum.size == 0:
        print(f"WARNING: layer {layer_id} ({matrix_name}) has no positive ESD")
        continue

    ESD_BY_LAYER[layer_id] = spectrum
    LAYER_NAME_BY_ID[layer_id] = matrix_name

    pd.DataFrame(
        {
            "ascending_rank": np.arange(1, spectrum.size + 1),
            "eigenvalue": spectrum,
        }
    ).to_csv(
        OUTPUT_DIR / f"get_esd_layer_{layer_id}_{matrix_name}.csv",
        index=False,
    )

print(
    "extracted layer IDs:",
    sorted(ESD_BY_LAYER),
)

## **EDIT THIS CELL**

Choose the WeightWatcher layer ID and exactly how many largest eigenvalues to remove. Then rerun this cell and every cell below it.

`REMOVE_TOP_EIGENVALUES=0` fits the unmodified ESD. Use `1`, `2`, `3`, ..., `10` to remove the decaying upper tail one component at a time.

In [ ]:
# ======================= EDIT THIS CELL =======================

LAYER_ID = 1
REMOVE_TOP_EIGENVALUES = 10

# Plotting controls.
BIN_COUNTS = [16, 32, 64, 128]
KDE_BANDWIDTH = 0.25

# Sweep controls.
SWEEP_MAX_REMOVE = 15
MIN_RETAINED_EIGENVALUES = 20

# =============================================================

if LAYER_ID not in ESD_BY_LAYER:
    raise ValueError(
        f"LAYER_ID={LAYER_ID} is unavailable. "
        f"Choose one of {sorted(ESD_BY_LAYER)} from the table above."
    )

FULL_ESD = ESD_BY_LAYER[LAYER_ID].copy()
MATRIX_NAME = LAYER_NAME_BY_ID[LAYER_ID]

if REMOVE_TOP_EIGENVALUES < 0:
    raise ValueError("REMOVE_TOP_EIGENVALUES must be nonnegative")
if REMOVE_TOP_EIGENVALUES >= FULL_ESD.size:
    raise ValueError(
        "REMOVE_TOP_EIGENVALUES must be smaller than the ESD size"
    )

if REMOVE_TOP_EIGENVALUES == 0:
    RETAINED_ESD = FULL_ESD.copy()
    REMOVED_ESD = np.array([], dtype=float)
else:
    RETAINED_ESD = FULL_ESD[:-REMOVE_TOP_EIGENVALUES].copy()
    REMOVED_ESD = FULL_ESD[-REMOVE_TOP_EIGENVALUES:].copy()

if RETAINED_ESD.size < MIN_RETAINED_EIGENVALUES:
    raise ValueError(
        f"Only {RETAINED_ESD.size} eigenvalues remain; "
        f"need at least {MIN_RETAINED_EIGENVALUES}"
    )

print(f"selected layer: {LAYER_ID} ({MATRIX_NAME})")
print(f"full ESD size: {FULL_ESD.size}")
print(f"removed from top: {REMOVE_TOP_EIGENVALUES}")
print(f"retained ESD size: {RETAINED_ESD.size}")
print(
    "retained range:",
    (float(RETAINED_ESD.min()), float(RETAINED_ESD.max())),
)

if REMOVED_ESD.size:
    removed_table = pd.DataFrame(
        {
            "removed_order_largest_first": np.arange(
                1,
                REMOVED_ESD.size + 1,
            ),
            "eigenvalue": REMOVED_ESD[::-1],
        }
    )
    display(removed_table)

## Robust ESD plotting helpers

Three independent views are used:

- **occupancy-colored log bins**: the point color is the exact number of eigenvalues in the bin;
- **log-space KDE**: smoothing is performed on `log(eigenvalue)` and transformed back to a density in eigenvalue space;
- **unbinned CCDF**: no histogram or smoothing parameters enter at all.

In [ ]:
def log_binned_density(
    values: np.ndarray,
    n_bins: int,
) -> pd.DataFrame:
    values = np.sort(
        np.asarray(values, dtype=float).reshape(-1)
    )
    values = values[
        np.isfinite(values)
        & (values > 0)
    ]
    if values.size < 2:
        raise ValueError("Need at least two positive values")

    lower = max(
        np.nextafter(values.min(), 0.0),
        np.finfo(float).tiny,
    )
    upper = np.nextafter(values.max(), np.inf)
    edges = np.geomspace(lower, upper, int(n_bins) + 1)

    counts, edges = np.histogram(values, bins=edges)
    widths = np.diff(edges)
    centers = np.sqrt(edges[:-1] * edges[1:])
    density = counts / (values.size * widths)

    return pd.DataFrame(
        {
            "left": edges[:-1],
            "right": edges[1:],
            "center": centers,
            "width": widths,
            "count": counts,
            "density": density,
        }
    )


def log_space_kde(
    values: np.ndarray,
    bandwidth: float,
    n_grid: int = 800,
) -> tuple[np.ndarray, np.ndarray]:
    values = np.sort(
        np.asarray(values, dtype=float).reshape(-1)
    )
    values = values[
        np.isfinite(values)
        & (values > 0)
    ]
    if values.size < 3:
        raise ValueError("Need at least three positive values for KDE")

    log_values = np.log(values)
    kde = gaussian_kde(
        log_values,
        bw_method=float(bandwidth),
    )
    grid = np.geomspace(
        values.min(),
        values.max(),
        int(n_grid),
    )

    # If Y=log(X), then f_X(x)=f_Y(log(x))/x.
    density = kde(np.log(grid)) / grid
    return grid, density


def empirical_ccdf(
    values: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    values = np.sort(
        np.asarray(values, dtype=float).reshape(-1)
    )
    values = values[
        np.isfinite(values)
        & (values > 0)
    ]
    survival = (
        values.size - np.arange(values.size)
    ) / values.size
    return values, survival

## Full ESD with several logarithmic bin sizes

The original ESD remains on the plot. The region to the right of the retained maximum is the manually removed upper tail. Bin points are colored by occupancy so one-eigenvalue and two-eigenvalue ridges are visible rather than mistaken for separate densities.

In [ ]:
n_panels = len(BIN_COUNTS)
n_columns = 2
n_rows = int(np.ceil(n_panels / n_columns))

fig, axes = plt.subplots(
    n_rows,
    n_columns,
    figsize=(14, 5 * n_rows),
    constrained_layout=True,
)
axes = np.atleast_1d(axes).ravel()

for axis, n_bins in zip(axes, BIN_COUNTS):
    histogram = log_binned_density(FULL_ESD, n_bins)
    histogram.to_csv(
        OUTPUT_DIR
        / f"layer_{LAYER_ID}_{MATRIX_NAME}_log_bins_{n_bins}.csv",
        index=False,
    )

    visible = histogram.loc[
        (histogram["count"] > 0)
        & np.isfinite(histogram["density"])
        & (histogram["density"] > 0)
    ]
    maximum_count = int(visible["count"].max())

    color_norm = (
        LogNorm(vmin=1, vmax=maximum_count)
        if maximum_count > 1
        else None
    )
    points = axis.scatter(
        visible["center"],
        visible["density"],
        c=visible["count"],
        cmap="viridis",
        norm=color_norm,
        s=30 + 12 * np.sqrt(visible["count"]),
        edgecolors="none",
    )

    if REMOVED_ESD.size:
        axis.axvline(
            RETAINED_ESD.max(),
            linestyle="--",
            linewidth=1.5,
            label=(
                f"retain through {RETAINED_ESD.max():.4g}; "
                f"remove {REMOVE_TOP_EIGENVALUES}"
            ),
        )
        axis.axvspan(
            RETAINED_ESD.max(),
            FULL_ESD.max(),
            alpha=0.10,
        )

    axis.set_xscale("log")
    axis.set_yscale("log")
    axis.set_xlabel(r"eigenvalue $\lambda$")
    axis.set_ylabel(r"density $\widehat{\rho}(\lambda)$")
    axis.set_title(f"{n_bins} logarithmic bins")
    axis.grid(True, which="both", alpha=0.20)
    axis.legend(fontsize=8)
    fig.colorbar(
        points,
        ax=axis,
        pad=0.01,
        label="eigenvalues per bin",
    )

for axis in axes[n_panels:]:
    axis.set_visible(False)

fig.suptitle(
    f"{MATRIX_NAME}, layer_id={LAYER_ID}: "
    f"full get_ESD spectrum; remove top "
    f"{REMOVE_TOP_EIGENVALUES}",
    fontsize=14,
)
figure_path = (
    OUTPUT_DIR
    / f"layer_{LAYER_ID}_{MATRIX_NAME}_multi_bin_esd.png"
)
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()
print("saved:", figure_path)

## Smoothed density and unbinned CCDF

The KDE panel shows both the complete spectrum and the retained spectrum. The CCDF panel is the least assumption-dependent view and marks every removed eigenvalue explicitly.

In [ ]:
full_grid, full_kde = log_space_kde(
    FULL_ESD,
    KDE_BANDWIDTH,
)
retained_grid, retained_kde = log_space_kde(
    RETAINED_ESD,
    KDE_BANDWIDTH,
)

full_x, full_ccdf = empirical_ccdf(FULL_ESD)
retained_x, retained_ccdf = empirical_ccdf(RETAINED_ESD)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(15, 5.5),
    constrained_layout=True,
)

axes[0].loglog(
    full_grid,
    full_kde,
    linewidth=2,
    label="full ESD, log-KDE",
)
axes[0].loglog(
    retained_grid,
    retained_kde,
    linewidth=2,
    label=(
        f"retained after removing top "
        f"{REMOVE_TOP_EIGENVALUES}"
    ),
)
if REMOVED_ESD.size:
    axes[0].axvline(
        RETAINED_ESD.max(),
        linestyle="--",
        label="manual upper cutoff",
    )
axes[0].set_xlabel(r"eigenvalue $\lambda$")
axes[0].set_ylabel(r"smoothed density $\rho(\lambda)$")
axes[0].set_title(
    f"log-space KDE, bandwidth={KDE_BANDWIDTH}"
)
axes[0].grid(True, which="both", alpha=0.20)
axes[0].legend()

axes[1].loglog(
    full_x,
    full_ccdf,
    ".",
    markersize=5,
    label="full ESD",
)
axes[1].loglog(
    retained_x,
    retained_ccdf,
    ".",
    markersize=5,
    label="retained ESD",
)
if REMOVED_ESD.size:
    removed_survival = (
        FULL_ESD.size
        - np.searchsorted(FULL_ESD, REMOVED_ESD)
    ) / FULL_ESD.size
    axes[1].scatter(
        REMOVED_ESD,
        removed_survival,
        marker="x",
        s=70,
        linewidths=2,
        label="removed top eigenvalues",
    )
axes[1].set_xlabel(r"eigenvalue $\lambda$")
axes[1].set_ylabel(r"$P(\Lambda \geq \lambda)$")
axes[1].set_title("empirical unbinned CCDF")
axes[1].grid(True, which="both", alpha=0.20)
axes[1].legend()

figure_path = (
    OUTPUT_DIR
    / f"layer_{LAYER_ID}_{MATRIX_NAME}_kde_ccdf.png"
)
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()
print("saved:", figure_path)

## Fit the manually retained ESD with `powerlaw`

Two MLE fits are reported:

- **all retained eigenvalues**: fixes `xmin` to the smallest retained eigenvalue, so the complete spectrum remaining after manual top removal is fitted;
- **automatic xmin**: lets `powerlaw` select where a smaller power-law tail begins.

The likelihood-ratio comparison is `power_law` versus `truncated_power_law`. A positive `R` favors the ordinary power law; a negative `R` favors the truncated power law. Interpret the sign only when `p` is small enough to be informative.

In [ ]:
def safe_float(value) -> float:
    try:
        result = float(value)
    except (TypeError, ValueError):
        return float("nan")
    return result if np.isfinite(result) else float("nan")


def summarize_powerlaw_fit(
    fit,
    *,
    mode: str,
    input_values: np.ndarray,
    removed_top: int,
) -> dict[str, object]:
    power_law_model = fit.power_law
    xmin = safe_float(
        getattr(
            power_law_model,
            "xmin",
            getattr(fit, "xmin", np.nan),
        )
    )
    n_tail = (
        int(np.count_nonzero(input_values >= xmin))
        if np.isfinite(xmin)
        else 0
    )

    comparison_R = np.nan
    comparison_p = np.nan
    comparison_error = ""
    try:
        comparison_R, comparison_p = fit.distribution_compare(
            "power_law",
            "truncated_power_law",
        )
    except Exception as exception:
        comparison_error = (
            f"{type(exception).__name__}: {exception}"
        )

    return {
        "layer_id": int(LAYER_ID),
        "matrix_name": MATRIX_NAME,
        "removed_top_eigenvalues": int(removed_top),
        "fit_mode": mode,
        "n_input": int(input_values.size),
        "n_tail": n_tail,
        "tail_fraction": (
            n_tail / input_values.size
            if input_values.size
            else np.nan
        ),
        "xmin": xmin,
        "alpha": safe_float(
            getattr(power_law_model, "alpha", np.nan)
        ),
        "sigma": safe_float(
            getattr(power_law_model, "sigma", np.nan)
        ),
        "D": safe_float(
            getattr(power_law_model, "D", np.nan)
        ),
        "R_power_law_vs_truncated": safe_float(comparison_R),
        "p_power_law_vs_truncated": safe_float(comparison_p),
        "comparison_error": comparison_error,
    }


with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    # This is the requested fit to the entire ESD after removing
    # REMOVE_TOP_EIGENVALUES from the top.
    FULL_RETAINED_FIT = powerlaw.Fit(
        RETAINED_ESD,
        discrete=False,
        xmin=float(RETAINED_ESD.min()),
        verbose=False,
    )

    # This second fit reports where powerlaw's xmin estimator places a
    # smaller conventional tail within the retained data.
    AUTO_XMIN_FIT = powerlaw.Fit(
        RETAINED_ESD,
        discrete=False,
        verbose=False,
    )

manual_fit_summary = pd.DataFrame(
    [
        summarize_powerlaw_fit(
            FULL_RETAINED_FIT,
            mode="all_retained_eigenvalues",
            input_values=RETAINED_ESD,
            removed_top=REMOVE_TOP_EIGENVALUES,
        ),
        summarize_powerlaw_fit(
            AUTO_XMIN_FIT,
            mode="automatic_xmin_tail",
            input_values=RETAINED_ESD,
            removed_top=REMOVE_TOP_EIGENVALUES,
        ),
    ]
)

manual_fit_summary.to_csv(
    OUTPUT_DIR
    / f"layer_{LAYER_ID}_{MATRIX_NAME}_manual_fit_summary.csv",
    index=False,
)
display(manual_fit_summary)

## Overlay the MLE fits on the unbinned CCDF and smoothed density

The full-retained model is normalized over every retained eigenvalue. The automatic-`xmin` model is scaled by the empirical fraction of retained eigenvalues above its fitted `xmin`.

In [ ]:
def power_law_ccdf(
    grid: np.ndarray,
    *,
    alpha: float,
    xmin: float,
    tail_fraction: float = 1.0,
) -> np.ndarray:
    result = np.full_like(grid, np.nan, dtype=float)
    valid = grid >= xmin
    result[valid] = (
        tail_fraction
        * (grid[valid] / xmin) ** (1.0 - alpha)
    )
    return result


def power_law_pdf(
    grid: np.ndarray,
    *,
    alpha: float,
    xmin: float,
    tail_fraction: float = 1.0,
) -> np.ndarray:
    result = np.full_like(grid, np.nan, dtype=float)
    valid = grid >= xmin
    result[valid] = (
        tail_fraction
        * (alpha - 1.0)
        / xmin
        * (grid[valid] / xmin) ** (-alpha)
    )
    return result


full_summary = manual_fit_summary.loc[
    manual_fit_summary["fit_mode"] == "all_retained_eigenvalues"
].iloc[0]
auto_summary = manual_fit_summary.loc[
    manual_fit_summary["fit_mode"] == "automatic_xmin_tail"
].iloc[0]

plot_grid = np.geomspace(
    RETAINED_ESD.min(),
    RETAINED_ESD.max(),
    1000,
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(15, 5.5),
    constrained_layout=True,
)

axes[0].loglog(
    retained_grid,
    retained_kde,
    linewidth=2,
    label="retained ESD log-KDE",
)
axes[0].loglog(
    plot_grid,
    power_law_pdf(
        plot_grid,
        alpha=float(full_summary["alpha"]),
        xmin=float(full_summary["xmin"]),
        tail_fraction=1.0,
    ),
    linewidth=2,
    label=(
        "full retained PL: "
        f"alpha={full_summary['alpha']:.3f}"
    ),
)
axes[0].loglog(
    plot_grid,
    power_law_pdf(
        plot_grid,
        alpha=float(auto_summary["alpha"]),
        xmin=float(auto_summary["xmin"]),
        tail_fraction=float(auto_summary["tail_fraction"]),
    ),
    linestyle="--",
    linewidth=2,
    label=(
        "auto-xmin PL: "
        f"alpha={auto_summary['alpha']:.3f}, "
        f"xmin={auto_summary['xmin']:.3g}"
    ),
)
axes[0].axvline(
    float(auto_summary["xmin"]),
    linestyle=":",
    label="automatic xmin",
)
axes[0].set_xlabel(r"eigenvalue $\lambda$")
axes[0].set_ylabel(r"density $\rho(\lambda)$")
axes[0].set_title("smoothed density with MLE overlays")
axes[0].grid(True, which="both", alpha=0.20)
axes[0].legend(fontsize=8)

axes[1].loglog(
    retained_x,
    retained_ccdf,
    ".",
    markersize=5,
    label="retained empirical CCDF",
)
axes[1].loglog(
    plot_grid,
    power_law_ccdf(
        plot_grid,
        alpha=float(full_summary["alpha"]),
        xmin=float(full_summary["xmin"]),
        tail_fraction=1.0,
    ),
    linewidth=2,
    label=(
        "full retained PL: "
        f"alpha={full_summary['alpha']:.3f}, "
        f"D={full_summary['D']:.3f}"
    ),
)
axes[1].loglog(
    plot_grid,
    power_law_ccdf(
        plot_grid,
        alpha=float(auto_summary["alpha"]),
        xmin=float(auto_summary["xmin"]),
        tail_fraction=float(auto_summary["tail_fraction"]),
    ),
    linestyle="--",
    linewidth=2,
    label=(
        "auto-xmin PL: "
        f"alpha={auto_summary['alpha']:.3f}, "
        f"D={auto_summary['D']:.3f}"
    ),
)
axes[1].axvline(
    float(auto_summary["xmin"]),
    linestyle=":",
    label="automatic xmin",
)
axes[1].set_xlabel(r"eigenvalue $\lambda$")
axes[1].set_ylabel("CCDF")
axes[1].set_title("unbinned CCDF with MLE overlays")
axes[1].grid(True, which="both", alpha=0.20)
axes[1].legend(fontsize=8)

figure_path = (
    OUTPUT_DIR
    / f"layer_{LAYER_ID}_{MATRIX_NAME}_powerlaw_overlays.png"
)
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()
print("saved:", figure_path)

## Systematic 0, 1, 2, ... top-eigenvalue removal sweep

This cell repeats both MLE fits for every removal count through `SWEEP_MAX_REMOVE`. The resulting table makes the breakpoint visible: look for a substantial drop in KS `D` followed by relative stability of `alpha`, rather than choosing a removal count only because `D` decreases monotonically.

In [ ]:
def fit_after_removing_top(
    full_esd: np.ndarray,
    remove_top: int,
) -> list[dict[str, object]]:
    if remove_top < 0:
        raise ValueError("remove_top must be nonnegative")

    retained = (
        full_esd.copy()
        if remove_top == 0
        else full_esd[:-remove_top].copy()
    )
    if retained.size < MIN_RETAINED_EIGENVALUES:
        return []

    results = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        try:
            complete_fit = powerlaw.Fit(
                retained,
                discrete=False,
                xmin=float(retained.min()),
                verbose=False,
            )
            complete_summary = summarize_powerlaw_fit(
                complete_fit,
                mode="all_retained_eigenvalues",
                input_values=retained,
                removed_top=remove_top,
            )
            complete_summary["error"] = ""
            results.append(complete_summary)
        except Exception as exception:
            results.append(
                {
                    "layer_id": int(LAYER_ID),
                    "matrix_name": MATRIX_NAME,
                    "removed_top_eigenvalues": int(remove_top),
                    "fit_mode": "all_retained_eigenvalues",
                    "n_input": int(retained.size),
                    "error": (
                        f"{type(exception).__name__}: {exception}"
                    ),
                }
            )

        try:
            tail_fit = powerlaw.Fit(
                retained,
                discrete=False,
                verbose=False,
            )
            tail_summary = summarize_powerlaw_fit(
                tail_fit,
                mode="automatic_xmin_tail",
                input_values=retained,
                removed_top=remove_top,
            )
            tail_summary["error"] = ""
            results.append(tail_summary)
        except Exception as exception:
            results.append(
                {
                    "layer_id": int(LAYER_ID),
                    "matrix_name": MATRIX_NAME,
                    "removed_top_eigenvalues": int(remove_top),
                    "fit_mode": "automatic_xmin_tail",
                    "n_input": int(retained.size),
                    "error": (
                        f"{type(exception).__name__}: {exception}"
                    ),
                }
            )

    return results


maximum_remove = min(
    int(SWEEP_MAX_REMOVE),
    int(FULL_ESD.size - MIN_RETAINED_EIGENVALUES),
)

sweep_rows = []
for remove_top in range(maximum_remove + 1):
    sweep_rows.extend(
        fit_after_removing_top(
            FULL_ESD,
            remove_top,
        )
    )

trim_sweep = pd.DataFrame(sweep_rows)
trim_sweep.to_csv(
    OUTPUT_DIR
    / f"layer_{LAYER_ID}_{MATRIX_NAME}_top_removal_sweep.csv",
    index=False,
)

successful_sweep = trim_sweep.loc[
    trim_sweep.get("error", "").fillna("").eq("")
].copy()

display(
    successful_sweep[
        [
            column
            for column in (
                "removed_top_eigenvalues",
                "fit_mode",
                "n_input",
                "n_tail",
                "alpha",
                "sigma",
                "D",
                "xmin",
                "R_power_law_vs_truncated",
                "p_power_law_vs_truncated",
            )
            if column in successful_sweep.columns
        ]
    ]
)

In [ ]:
fig, axes = plt.subplots(
    2,
    3,
    figsize=(17, 10),
    constrained_layout=True,
)

plot_specs = [
    ("alpha", r"$\alpha$", False),
    ("D", "KS D", False),
    ("xmin", r"$x_{\min}$", True),
    ("n_tail", "number fitted", False),
    (
        "R_power_law_vs_truncated",
        "log-likelihood ratio R\nPL vs truncated PL",
        False,
    ),
    (
        "p_power_law_vs_truncated",
        "comparison p-value",
        True,
    ),
]

for axis, (metric, label, use_log_scale) in zip(
    axes.ravel(),
    plot_specs,
):
    for fit_mode, group in successful_sweep.groupby("fit_mode"):
        axis.plot(
            group["removed_top_eigenvalues"],
            group[metric],
            "o-",
            markersize=4,
            label=fit_mode,
        )

    axis.axvline(
        REMOVE_TOP_EIGENVALUES,
        linestyle="--",
        linewidth=1.3,
        label="current manual choice",
    )
    axis.set_xlabel("largest eigenvalues removed")
    axis.set_ylabel(label)
    axis.set_title(label)
    axis.grid(True, which="both", alpha=0.20)
    if use_log_scale:
        positive = successful_sweep[metric] > 0
        if positive.any():
            axis.set_yscale("log")
    axis.legend(fontsize=8)

fig.suptitle(
    f"{MATRIX_NAME}, layer_id={LAYER_ID}: "
    "systematic top-eigenvalue removal",
    fontsize=14,
)
figure_path = (
    OUTPUT_DIR
    / f"layer_{LAYER_ID}_{MATRIX_NAME}_top_removal_sweep.png"
)
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()
print("saved:", figure_path)

## Reading the result

For the manually selected `REMOVE_TOP_EIGENVALUES`:

- the `all_retained_eigenvalues` row is the requested MLE fit over the complete ESD after discarding those largest components;
- the `automatic_xmin_tail` row reports where the conventional tail estimator starts within that retained ESD;
- `D` measures the empirical-versus-model discrepancy;
- `R < 0` with a sufficiently small `p` favors `truncated_power_law` over `power_law`;
- the multi-bin and KDE figures show whether the apparent global slope survives a change in plotting method;
- the CCDF is the bin-free check.

To test another choice, change only `LAYER_ID` and `REMOVE_TOP_EIGENVALUES` in the marked edit cell, then rerun the cells below it.